In [ ]:
import pandas as pd
import numpy as np
import joblib
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. TẢI 3 MÔ HÌNH "BỘ NÃO" CỦA HỆ THỐNG
# ==========================================
clf_model = joblib.load('../models/classification/best_classification_model.pkl')
reg_model = joblib.load('../models/regression/best_regression_model.pkl')
demand_model = joblib.load('../models/demand/best_demand_model.pkl')

def smart_parking_checkin(student_id, entry_time_str, vehicle, usual_zone, rolling_avg, hist_overnight):
    """
    Hệ thống xử lý Logic khi sinh viên quẹt thẻ vào bãi đỗ.
    """
    entry_time = pd.to_datetime(entry_time_str)
    
    # --- BƯỚC 1: DỰ ĐOÁN TÌNH TRẠNG BÃI ĐỖ HIỆN TẠI ---
    demand_features = pd.DataFrame([{
        'hour': entry_time.hour,
        'day_of_week': entry_time.weekday(),
        'is_weekend': 1 if entry_time.weekday() >= 5 else 0,
        'is_morning': 1 if entry_time.hour < 12 else 0,
        'is_exam_week': 0 # Giả sử đang không phải tuần thi
    }])
    # BÍ QUYẾT: Ép thứ tự cột chuẩn xác tuyệt đối cho Demand Model
    demand_features = demand_features[demand_model.feature_names_in_]
    current_demand = int(demand_model.predict(demand_features)[0])
    
    # --- BƯỚC 2: PHÂN LOẠI HÀNH VI & DỰ ĐOÁN GIỜ VỀ ---
    clf_features = pd.DataFrame([{
        'entry_hour': entry_time.hour,
        'entry_minute': entry_time.minute,
        'day_of_week_num': entry_time.weekday(),
        'is_weekend': 1 if entry_time.weekday() >= 5 else 0,
        'is_morning': 1 if entry_time.hour < 12 else 0,
        'rolling_avg_duration': rolling_avg,
        'historical_overnight_count': hist_overnight,
        'vehicle_type_Motorbike': 1 if vehicle == 'Motorbike' else 0,
        'usual_zone_Zone_B': 1 if usual_zone == 'Zone_B' else 0,
        'usual_zone_Zone_C': 1 if usual_zone == 'Zone_C' else 0,
        'usual_zone_Zone_D': 1 if usual_zone == 'Zone_D' else 0,
        'is_exam_week': 0
    }])
    
    # BÍ QUYẾT: Ép thứ tự cột chuẩn xác tuyệt đối cho Classification/Regression Model
    clf_features = clf_features[clf_model.feature_names_in_]
    
    # Mô hình AI đưa ra quyết định
    predicted_behavior = clf_model.predict(clf_features)[0]
    raw_predicted_minutes = reg_model.predict(clf_features)[0]
    
    # BƯỚC 3: ĐIỀU CHỈNH LOGIC (Two-Stage Recommendation)
    if predicted_behavior == 'Early_Return':
        estimated_minutes = min(raw_predicted_minutes, 210) # Tối đa 3.5 tiếng
        zone_recommend = "Khu A (Dành cho xe ra vào nhanh)"
    elif predicted_behavior == 'Full_Day':
        estimated_minutes = max(raw_predicted_minutes, 240) # Ít nhất 4 tiếng
        zone_recommend = "Khu B hoặc C (Đỗ lâu dài)"
    elif predicted_behavior == 'Overnight':
        estimated_minutes = max(raw_predicted_minutes, 720) # Ít nhất 12 tiếng
        zone_recommend = "Khu D (Khu vực an ninh qua đêm)"
    else:
        estimated_minutes = raw_predicted_minutes
        zone_recommend = "Khu D (Bãi xe dài ngày)"
        
    estimated_exit = entry_time + pd.Timedelta(minutes=estimated_minutes)
    
    # --- BƯỚC 4: IN VÉ XE THÔNG MINH ---
    print("="*50)
    print("🚗 SMART PARKING SYSTEM - VÉ XE ĐIỆN TỬ 🚗")
    print("="*50)
    print(f"👤 Mã Sinh Viên      : {student_id}")
    print(f"⏰ Thời gian vào      : {entry_time.strftime('%H:%M - %d/%m/%Y')}")
    print(f"📊 Dự kiến bãi đỗ     : Có khoảng {current_demand} xe đang trong bãi")
    print("-" * 50)
    print("🧠 HỆ THỐNG AI DỰ ĐOÁN:")
    print(f"   ▶ Hành vi đỗ xe    : {predicted_behavior}")
    print(f"   ▶ Thời gian rời bãi: Khoảng {estimated_exit.strftime('%H:%M')} (đỗ ~{int(estimated_minutes/60)} tiếng {int(estimated_minutes%60)} phút)")
    print(f"   ▶ Vị trí đề xuất   : {zone_recommend}")
    print("="*50)

# Chạy thử nghiệm với 1 sinh viên IT vào lúc 7h15 sáng thứ Hai
smart_parking_checkin(
    student_id="SV2023_CNTT",
    entry_time_str="2023-11-20 07:15:00",
    vehicle="Motorbike",
    usual_zone="Zone_A",
    rolling_avg=260,     # Sinh viên này bình thường hay đỗ khoảng 4 tiếng rưỡi
    hist_overnight=0     # Chưa từng đỗ qua đêm
)

ValueError: The feature names should match those that were passed during fit.
Feature names must be in the same order as they were in fit.
